# 3D Grad-CAM Generation and Explanation Faithfulness Evaluation

## Objective

This notebook generates 3D Grad-CAM attention maps for the Run 2 3D ResNet-18 PDAC classifier and quantitatively evaluates whether the model's attention actually localizes to the annotated PDAC lesion.

This is the explainability half of the thesis's trustworthiness evaluation (calibration was the other half). The two ask different questions:

- Calibration: are the model's stated confidences reliable?
- Grad-CAM faithfulness: when the model says "PDAC," is it looking at the tumor, or somewhere else?

## Methodology

1. Load the Run 2 best checkpoint (`best_model.pt`) and rebuild the identical architecture used in training.
2. For each **PDAC-positive test case**, generate a 3D Grad-CAM heatmap via forward + backward hooks on `model.layer4` (the last residual stage before global average pooling -- the standard, most-cited target layer for ResNet-family Grad-CAM).
3. Compare the heatmap against the ground-truth tumor annotation (`mask == 1`) using:
   - **IoU** and **Dice** (require binarizing the continuous heatmap -- computed across a *range* of thresholds, not a single arbitrary cutoff, since this choice materially affects the result).
   - **Pointing Game** (threshold-independent: does the single highest-activation voxel fall inside the tumor mask?).
4. Report summary statistics with bootstrap confidence intervals, and save qualitative example figures.

## Key design decisions and why

- **Backprop target is the raw logit, not the sigmoid probability.** Backpropagating from a saturated probability (e.g. 0.999) produces a near-zero gradient regardless of what the network actually attended to, which would silently degrade Grad-CAM specifically on the model's most confident cases. The logit does not saturate this way.
- **Evaluation is restricted to PDAC-positive test cases.** Non-PDAC cases have no tumor region (`mask == 1` is absent by construction, confirmed during dataset auditing), so there is nothing to localize against.
- **No raw/cropped alignment risk.** This notebook uses the already-*processed* masks -- the same crop, same voxel grid as the images the model actually received as input. The pancreas-centered ROI crop alignment questions investigated earlier (Notebook 18) concerned raw pre-crop volumes and do not apply here.
- **Known limitation carried over from Notebook 18**: two PDAC cases in the full dataset had a lesion extending beyond the lateral ROI boundary (92.9%-94.2% tumor voxel retention after cropping). If either case falls in the test split, its ground-truth mask reflects only the retained (cropped) tumor extent -- consistent with what the classifier itself saw, but worth noting explicitly if either case appears among the results below.
- **GPU required, but far lighter than training.** Grad-CAM needs one forward + one backward pass per case (no optimizer step, no multiple epochs), so this should complete in a few minutes on a T4 for ~100 PDAC test cases, versus hours if attempted on CPU.


In [ ]:
# ============================================================
# CELL 1 — IMPORTS AND CONFIGURATION
# ============================================================

from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import torch
import torch.nn as nn
import torch.nn.functional as F
from torchvision.models.video import r3d_18

# ------------------------------------------------------------
# Kaggle paths — UPDATE THESE for your session.
# ------------------------------------------------------------
# The processed-images/masks dataset is the same one used for training.
# best_model.pt needs to be attached as its own small Kaggle dataset:
# upload checkpoints_run2/best_model.pt (downloaded earlier from the
# training notebook's Output tab) as a new Kaggle dataset, then attach
# both datasets to this notebook via "Add Input".

PROCESSED_DIR = Path(
    "/kaggle/input/datasets/shahriarahmed66/panorama-processed-dataset"
)
CHECKPOINT_DIR = Path(
    "/kaggle/input/datasets/shahriarahmed66/panorama-run2-best-checkpoint"  # <-- adjust to your actual attached dataset slug
)
CHECKPOINT_PATH = CHECKPOINT_DIR / "best_model.pt"

SPLIT_PATH = PROCESSED_DIR / "split_assignment.csv"

OUTPUT_DIR = Path("/kaggle/working/gradcam_results")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

SEED = 42
np.random.seed(SEED)
torch.manual_seed(SEED)

# Thresholds swept for IoU/Dice, applied to the min-max normalized CAM.
# 0.5 is reported as the headline value; the full sweep is reported too.
IOU_DICE_THRESHOLDS = [0.3, 0.4, 0.5, 0.6, 0.7]
HEADLINE_THRESHOLD = 0.5

N_BOOTSTRAP = 2000

# Save full normalized CAM volumes only for a handful of exemplar cases
# (best/median/worst by Dice) rather than all ~100 cases, to keep output
# size manageable -- set True if you want every case's CAM saved.
SAVE_ALL_CAM_VOLUMES = False

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print("=" * 70)
print("GRAD-CAM EVALUATION — CONFIGURATION")
print("=" * 70)
print("Device            :", DEVICE)
print("Processed dir     :", PROCESSED_DIR)
print("Checkpoint path   :", CHECKPOINT_PATH)
print("  exists          :", CHECKPOINT_PATH.exists())
print("Output dir        :", OUTPUT_DIR)
print("IoU/Dice thresholds:", IOU_DICE_THRESHOLDS)
print("Headline threshold :", HEADLINE_THRESHOLD)

if DEVICE.type != "cuda":
    print(
        "\nWARNING: no GPU detected. Grad-CAM generation for ~100 cases will "
        "be slow on CPU (each case needs a full forward + backward pass). "
        "Enable a GPU accelerator before running the main loop below."
    )


## Load and filter the split

Only PDAC-positive test cases are used for quantitative Grad-CAM evaluation. The same path corrections established during training (backslash-to-forward-slash, and the extra nested `images/images/` / `masks/masks/` level from how the dataset zip was built) are applied here identically.

In [ ]:
# ============================================================
# CELL 2 — LOAD SPLIT AND FILTER TO PDAC-POSITIVE TEST CASES
# ============================================================

split_df = pd.read_csv(SPLIT_PATH)

# Same two corrections verified during training (Notebook 19/20):
# backslash -> forward slash, and the extra nested images/masks level
# from how images.zip/masks.zip were built with 7-Zip.
split_df["image_path"] = "images/" + (
    split_df["image_path"].astype(str).str.replace("\\", "/", regex=False)
)
split_df["mask_path"] = "masks/" + (
    split_df["mask_path"].astype(str).str.replace("\\", "/", regex=False)
)

test_df = split_df[split_df["split"].str.lower() == "test"].reset_index(drop=True)
pdac_test_df = test_df[test_df["diagnosis"] == "PDAC"].reset_index(drop=True)

print("=" * 70)
print("TEST SET FILTERED TO PDAC-POSITIVE CASES")
print("=" * 70)
print(f"Total test cases      : {len(test_df)}")
print(f"PDAC-positive cases   : {len(pdac_test_df)}")
print(f"non-PDAC cases (excluded from Grad-CAM evaluation): "
      f"{len(test_df) - len(pdac_test_df)}")

# Path sanity check before the main loop.
sample_image_path = PROCESSED_DIR / pdac_test_df.iloc[0]["image_path"]
sample_mask_path = PROCESSED_DIR / pdac_test_df.iloc[0]["mask_path"]
print(f"\nSample image path resolves: {sample_image_path.exists()} ({sample_image_path})")
print(f"Sample mask path resolves : {sample_mask_path.exists()} ({sample_mask_path})")

assert sample_image_path.exists() and sample_mask_path.exists(), (
    "Path resolution failed -- check PROCESSED_DIR and the dataset attachment "
    "before proceeding to the (GPU-consuming) main loop below."
)
print("\n✓ Paths resolve correctly.")


## Rebuild the model architecture and load the Run 2 checkpoint

The architecture must exactly match what was trained: `r3d_18` with the stem's first conv adapted from 3 input channels to 1, and the classification head replaced with a single logit. The backbone is built with `pretrained=False` here -- the pretrained Kinetics-400 weights are irrelevant for inference since every weight is about to be overwritten by the checkpoint anyway, and skipping the download avoids needing to enable the Kaggle Internet toggle for this notebook at all.

In [ ]:
# ============================================================
# CELL 3 — REBUILD ARCHITECTURE AND LOAD CHECKPOINT
# ============================================================

def build_model(pretrained=False):
    model = r3d_18(weights=None)  # weights irrelevant here -- overwritten below

    old_conv = model.stem[0]
    new_conv = nn.Conv3d(
        in_channels=1, out_channels=old_conv.out_channels,
        kernel_size=old_conv.kernel_size, stride=old_conv.stride,
        padding=old_conv.padding, bias=False,
    )
    model.stem[0] = new_conv

    in_features = model.fc.in_features
    model.fc = nn.Linear(in_features, 1)
    return model


model = build_model()

checkpoint = torch.load(CHECKPOINT_PATH, map_location=DEVICE, weights_only=False)
model.load_state_dict(checkpoint["model_state"])
model = model.to(DEVICE)
model.eval()

print("=" * 70)
print("MODEL LOADED")
print("=" * 70)
print("Checkpoint epoch      :", checkpoint["epoch"])
print("Checkpoint val loss   :", checkpoint["best_val_loss"])
print("Checkpoint val metrics:", checkpoint.get("val_metrics"))
print("\n✓ Model rebuilt and Run 2 best checkpoint loaded, set to eval mode.")


## 3D Grad-CAM implementation

Standard Grad-CAM, adapted to 3D:

1. Register a forward hook on the target layer to capture its activations `A` (shape `[1, C, d, h, w]`, spatially downsampled relative to the input).
2. Register a full backward hook on the same layer to capture the gradient of the output score with respect to `A`.
3. Backpropagate from the **raw logit** (not the sigmoid probability -- see the note in the introduction).
4. Global-average-pool the gradients over the spatial dimensions to get one importance weight per channel: `alpha_k = mean(dY/dA_k)`.
5. Compute the weighted combination `L = ReLU(sum_k alpha_k * A_k)`.
6. Upsample `L` via trilinear interpolation back to the full input volume size `(128, 160, 192)`.
7. Min-max normalize `L` to `[0, 1]` per case, for thresholding and visualization.

In [ ]:
# ============================================================
# CELL 4 — 3D GRAD-CAM CLASS
# ============================================================

class GradCAM3D:
    def __init__(self, model, target_layer):
        self.model = model
        self.activations = None
        self.gradients = None
        self._forward_handle = target_layer.register_forward_hook(self._save_activation)
        self._backward_handle = target_layer.register_full_backward_hook(self._save_gradient)

    def _save_activation(self, module, input, output):
        self.activations = output.detach()

    def _save_gradient(self, module, grad_input, grad_output):
        self.gradients = grad_output[0].detach()

    def remove_hooks(self):
        self._forward_handle.remove()
        self._backward_handle.remove()

    def generate(self, volume):
        """
        volume: tensor of shape (1, 1, D, H, W), already on the correct device.
        Returns (cam, probability) where cam is a (D, H, W) numpy array
        normalized to [0, 1], and probability is the model's sigmoid output
        for this case (for reference/reporting, not used in CAM computation).
        """
        self.model.zero_grad()

        logit = self.model(volume)          # shape (1, 1)
        score = logit.squeeze()             # scalar raw logit -- backprop target
        probability = torch.sigmoid(logit).item()

        score.backward()

        # activations, gradients: (1, C, d, h, w)
        weights = self.gradients.mean(dim=(2, 3, 4), keepdim=True)   # (1, C, 1, 1, 1)
        cam = (weights * self.activations).sum(dim=1, keepdim=True)  # (1, 1, d, h, w)
        cam = F.relu(cam)

        target_size = volume.shape[2:]  # (D, H, W)
        cam = F.interpolate(cam, size=target_size, mode="trilinear", align_corners=False)
        cam = cam.squeeze().cpu().numpy()

        cam_min, cam_max = cam.min(), cam.max()
        if cam_max - cam_min > 1e-8:
            cam = (cam - cam_min) / (cam_max - cam_min)
        else:
            # Degenerate case: the layer produced no discriminative signal
            # for this input. Reported as an all-zero map rather than
            # silently dividing by ~0, and flagged in the results table.
            cam = np.zeros_like(cam)

        return cam, probability


target_layer = model.layer4
gradcam = GradCAM3D(model, target_layer)

print("✓ GradCAM3D initialized, hooked on model.layer4.")


## Sanity check on one case

Before running the full loop, generate and visualize a Grad-CAM for a single case to confirm the pipeline works end-to-end -- correct shapes, a heatmap that isn't degenerate (all-zero or uniform), and a plausible-looking overlay.

In [ ]:
# ============================================================
# CELL 5 — SANITY CHECK ON ONE CASE
# ============================================================

def load_volume_and_mask(row, processed_dir):
    image_path = processed_dir / row["image_path"]
    mask_path = processed_dir / row["mask_path"]

    volume = np.load(image_path).astype(np.float32)
    mask = np.load(mask_path)
    tumor_mask = (mask == 1)

    return volume, tumor_mask


sample_row = pdac_test_df.iloc[0]
sample_volume, sample_tumor_mask = load_volume_and_mask(sample_row, PROCESSED_DIR)

print("Study ID       :", sample_row["study_id"])
print("Volume shape   :", sample_volume.shape)
print("Tumor voxels   :", int(sample_tumor_mask.sum()))

volume_tensor = torch.from_numpy(sample_volume).unsqueeze(0).unsqueeze(0).to(DEVICE)

sample_cam, sample_probability = gradcam.generate(volume_tensor)

print("\nCAM shape      :", sample_cam.shape)
print("CAM min/max    :", sample_cam.min(), sample_cam.max())
print("Model probability for this case:", f"{sample_probability:.4f}")

if sample_cam.max() < 1e-6:
    print(
        "\nWARNING: CAM is degenerate (all zero) for this sample case. "
        "If this happens for most cases in the main loop below, revisit "
        "the target layer choice or check that gradients are flowing "
        "correctly through the hooked module."
    )

# ------------------------------------------------------------
# Visualize: CT, tumor mask overlay, CAM overlay -- axial slice
# through the tumor's center.
# ------------------------------------------------------------

tumor_coords = np.argwhere(sample_tumor_mask)
z_center = int(tumor_coords[:, 0].mean()) if len(tumor_coords) > 0 else sample_volume.shape[0] // 2

fig, axes = plt.subplots(1, 3, figsize=(15, 5))

axes[0].imshow(sample_volume[z_center], cmap="gray")
axes[0].set_title(f"CT — z={z_center}")
axes[0].axis("off")

axes[1].imshow(sample_volume[z_center], cmap="gray")
axes[1].contour(sample_tumor_mask[z_center], colors="lime", linewidths=1.5)
axes[1].set_title("Tumor mask overlay")
axes[1].axis("off")

axes[2].imshow(sample_volume[z_center], cmap="gray")
axes[2].imshow(sample_cam[z_center], cmap="jet", alpha=0.5)
axes[2].contour(sample_tumor_mask[z_center], colors="lime", linewidths=1.5)
axes[2].set_title(f"Grad-CAM overlay (prob={sample_probability:.3f})")
axes[2].axis("off")

plt.suptitle(f"Sanity check — {sample_row['study_id']}")
plt.tight_layout()
plt.show()


## Evaluation metrics

**IoU** and **Dice** require binarizing the normalized CAM at some threshold; both are computed across the full `IOU_DICE_THRESHOLDS` sweep, not just one value.

**Pointing Game** is threshold-independent: it checks only whether the single highest-activation voxel in the CAM falls inside the tumor mask.

In [ ]:
# ============================================================
# CELL 6 — METRIC FUNCTIONS
# ============================================================

def compute_iou(cam_binary, mask_binary):
    intersection = np.logical_and(cam_binary, mask_binary).sum()
    union = np.logical_or(cam_binary, mask_binary).sum()
    return intersection / union if union > 0 else np.nan


def compute_dice(cam_binary, mask_binary):
    intersection = np.logical_and(cam_binary, mask_binary).sum()
    denom = cam_binary.sum() + mask_binary.sum()
    return (2 * intersection) / denom if denom > 0 else np.nan


def pointing_game_hit(cam, mask_binary):
    peak_index = np.unravel_index(np.argmax(cam), cam.shape)
    return bool(mask_binary[peak_index])


print("✓ IoU, Dice, and Pointing Game functions defined.")


## Main loop — generate Grad-CAM and compute metrics for every PDAC test case

This is the GPU-consuming step. Each case requires one forward pass and one backward pass through the full 3D network.

In [ ]:
# ============================================================
# CELL 7 — MAIN LOOP: GRAD-CAM + METRICS FOR ALL PDAC TEST CASES
# ============================================================

results = []
saved_cam_volumes = {}  # study_id -> cam array, populated selectively below

print("=" * 70)
print(f"GENERATING GRAD-CAM FOR {len(pdac_test_df)} PDAC TEST CASES")
print("=" * 70)

for i, row in pdac_test_df.iterrows():
    study_id = row["study_id"]

    volume, tumor_mask = load_volume_and_mask(row, PROCESSED_DIR)
    tumor_voxel_count = int(tumor_mask.sum())

    if tumor_voxel_count == 0:
        # Should not occur for a PDAC-labeled case with an intact mask,
        # but guard against it rather than silently skewing metrics.
        print(f"  [{i+1}/{len(pdac_test_df)}] {study_id}: SKIPPED (no tumor voxels in mask)")
        continue

    volume_tensor = torch.from_numpy(volume).unsqueeze(0).unsqueeze(0).to(DEVICE)
    cam, probability = gradcam.generate(volume_tensor)

    row_result = {
        "study_id": study_id,
        "probability": probability,
        "tumor_voxel_count": tumor_voxel_count,
        "cam_degenerate": bool(cam.max() < 1e-6),
        "pointing_game_hit": pointing_game_hit(cam, tumor_mask),
    }

    for threshold in IOU_DICE_THRESHOLDS:
        cam_binary = cam >= threshold
        row_result[f"iou_t{threshold}"] = compute_iou(cam_binary, tumor_mask)
        row_result[f"dice_t{threshold}"] = compute_dice(cam_binary, tumor_mask)

    results.append(row_result)

    if SAVE_ALL_CAM_VOLUMES:
        saved_cam_volumes[study_id] = cam

    if (i + 1) % 20 == 0 or (i + 1) == len(pdac_test_df):
        print(f"  Processed {i+1}/{len(pdac_test_df)}")

gradcam.remove_hooks()  # done generating -- detach hooks

results_df = pd.DataFrame(results)

print("\n" + "=" * 70)
print("GRAD-CAM GENERATION COMPLETE")
print("=" * 70)
print(f"Cases evaluated  : {len(results_df)}")
print(f"Degenerate CAMs  : {int(results_df['cam_degenerate'].sum())} "
      f"({'none -- good' if results_df['cam_degenerate'].sum() == 0 else 'investigate before trusting results'})")

display(results_df.head())


## Threshold sensitivity — headline results

Mean IoU and Dice are reported across the full threshold sweep, so the headline numbers (at `HEADLINE_THRESHOLD`) can be seen in context rather than as an isolated, unexplained choice. Pointing Game accuracy is a single number since it doesn't depend on a threshold.

In [ ]:
# ============================================================
# CELL 8 — THRESHOLD SENSITIVITY SUMMARY
# ============================================================

print("=" * 70)
print("IoU / DICE ACROSS THRESHOLDS (mean over all evaluated PDAC test cases)")
print("=" * 70)

threshold_summary = []
for threshold in IOU_DICE_THRESHOLDS:
    mean_iou = results_df[f"iou_t{threshold}"].mean()
    mean_dice = results_df[f"dice_t{threshold}"].mean()
    threshold_summary.append({"threshold": threshold, "mean_iou": mean_iou, "mean_dice": mean_dice})
    marker = "  <-- headline" if threshold == HEADLINE_THRESHOLD else ""
    print(f"  threshold={threshold:.1f}  mean_IoU={mean_iou:.4f}  mean_Dice={mean_dice:.4f}{marker}")

threshold_summary_df = pd.DataFrame(threshold_summary)

pointing_game_accuracy = results_df["pointing_game_hit"].mean()
print(f"\nPointing Game accuracy (threshold-independent): {pointing_game_accuracy:.4f} "
      f"({int(results_df['pointing_game_hit'].sum())}/{len(results_df)} cases)")

# Plot IoU/Dice vs threshold, so the sensitivity is visible at a glance.
fig, ax = plt.subplots(figsize=(7, 5))
ax.plot(threshold_summary_df["threshold"], threshold_summary_df["mean_iou"], "o-", label="Mean IoU")
ax.plot(threshold_summary_df["threshold"], threshold_summary_df["mean_dice"], "s-", label="Mean Dice")
ax.axvline(HEADLINE_THRESHOLD, color="gray", linestyle="--", alpha=0.6, label=f"Headline (T={HEADLINE_THRESHOLD})")
ax.set_xlabel("CAM binarization threshold")
ax.set_ylabel("Score")
ax.set_title("Grad-CAM Localization: IoU / Dice vs. Threshold")
ax.legend()
ax.grid(alpha=0.3)
plt.tight_layout()

threshold_plot_path = OUTPUT_DIR / "iou_dice_threshold_sensitivity.png"
plt.savefig(threshold_plot_path, dpi=150, bbox_inches="tight")
plt.show()
print("\nSaved:", threshold_plot_path)


## Bootstrap confidence intervals on the headline metrics

In [ ]:
# ============================================================
# CELL 9 — BOOTSTRAP CONFIDENCE INTERVALS
# ============================================================

def bootstrap_mean_ci(values, n_bootstrap=N_BOOTSTRAP, seed=SEED):
    values = np.asarray(values)
    values = values[~np.isnan(values)]
    point_estimate = values.mean()

    rng = np.random.default_rng(seed)
    n = len(values)
    boot_means = []
    for _ in range(n_bootstrap):
        idx = rng.integers(0, n, size=n)
        boot_means.append(values[idx].mean())

    ci_low, ci_high = np.percentile(boot_means, [2.5, 97.5])
    return point_estimate, ci_low, ci_high


print("=" * 70)
print(f"BOOTSTRAP 95% CONFIDENCE INTERVALS ({N_BOOTSTRAP} resamples)")
print("=" * 70)

headline_iou = results_df[f"iou_t{HEADLINE_THRESHOLD}"]
headline_dice = results_df[f"dice_t{HEADLINE_THRESHOLD}"]
pointing_hits = results_df["pointing_game_hit"].astype(float)

iou_point, iou_low, iou_high = bootstrap_mean_ci(headline_iou)
dice_point, dice_low, dice_high = bootstrap_mean_ci(headline_dice)
pg_point, pg_low, pg_high = bootstrap_mean_ci(pointing_hits)

print(f"Mean IoU  (T={HEADLINE_THRESHOLD}) : {iou_point:.4f}  (95% CI: [{iou_low:.4f}, {iou_high:.4f}])")
print(f"Mean Dice (T={HEADLINE_THRESHOLD}) : {dice_point:.4f}  (95% CI: [{dice_low:.4f}, {dice_high:.4f}])")
print(f"Pointing Game accuracy    : {pg_point:.4f}  (95% CI: [{pg_low:.4f}, {pg_high:.4f}])")

gradcam_summary_df = pd.DataFrame([
    {"metric": "IoU", "threshold": HEADLINE_THRESHOLD, "point_estimate": iou_point, "ci_low": iou_low, "ci_high": iou_high},
    {"metric": "Dice", "threshold": HEADLINE_THRESHOLD, "point_estimate": dice_point, "ci_low": dice_low, "ci_high": dice_high},
    {"metric": "Pointing Game", "threshold": "n/a", "point_estimate": pg_point, "ci_low": pg_low, "ci_high": pg_high},
])
display(gradcam_summary_df)


## Qualitative examples — best, median, and worst cases by Dice

Concrete visual examples matter as much as the aggregate numbers for a thesis chapter on explainability -- these let a reader judge for themselves whether the model's attention is plausible, not just take a summary statistic on faith.

In [ ]:
# ============================================================
# CELL 10 — QUALITATIVE EXAMPLES: BEST / MEDIAN / WORST BY DICE
# ============================================================

dice_col = f"dice_t{HEADLINE_THRESHOLD}"
sorted_results = results_df.dropna(subset=[dice_col]).sort_values(dice_col).reset_index(drop=True)

n = len(sorted_results)
example_indices = {
    "Worst": 0,
    "Median": n // 2,
    "Best": n - 1,
}

fig, axes = plt.subplots(len(example_indices), 3, figsize=(15, 5 * len(example_indices)))

for row_idx, (label, idx) in enumerate(example_indices.items()):
    example_row_summary = sorted_results.iloc[idx]
    study_id = example_row_summary["study_id"]
    dice_value = example_row_summary[dice_col]

    full_row = pdac_test_df[pdac_test_df["study_id"] == study_id].iloc[0]
    volume, tumor_mask = load_volume_and_mask(full_row, PROCESSED_DIR)

    volume_tensor = torch.from_numpy(volume).unsqueeze(0).unsqueeze(0).to(DEVICE)

    # Regenerate CAM for this specific case (hooks were removed after the
    # main loop, so re-attach them just for this visualization step).
    gradcam_viz = GradCAM3D(model, model.layer4)
    cam, probability = gradcam_viz.generate(volume_tensor)
    gradcam_viz.remove_hooks()

    tumor_coords = np.argwhere(tumor_mask)
    z_center = int(tumor_coords[:, 0].mean()) if len(tumor_coords) > 0 else volume.shape[0] // 2

    ax_ct, ax_mask, ax_cam = axes[row_idx]

    ax_ct.imshow(volume[z_center], cmap="gray")
    ax_ct.set_title(f"{label}: {study_id}\nCT — z={z_center}")
    ax_ct.axis("off")

    ax_mask.imshow(volume[z_center], cmap="gray")
    ax_mask.contour(tumor_mask[z_center], colors="lime", linewidths=1.5)
    ax_mask.set_title("Tumor mask")
    ax_mask.axis("off")

    ax_cam.imshow(volume[z_center], cmap="gray")
    ax_cam.imshow(cam[z_center], cmap="jet", alpha=0.5)
    ax_cam.contour(tumor_mask[z_center], colors="lime", linewidths=1.5)
    ax_cam.set_title(f"Grad-CAM (Dice={dice_value:.3f}, prob={probability:.3f})")
    ax_cam.axis("off")

    if SAVE_ALL_CAM_VOLUMES or label in ("Best", "Median", "Worst"):
        saved_cam_volumes[study_id] = cam

plt.tight_layout()

examples_plot_path = OUTPUT_DIR / "gradcam_qualitative_examples.png"
plt.savefig(examples_plot_path, dpi=150, bbox_inches="tight")
plt.show()
print("Saved:", examples_plot_path)


## Save all results

In [ ]:
# ============================================================
# CELL 11 — SAVE RESULTS
# ============================================================

per_case_path = OUTPUT_DIR / "gradcam_per_case_results.csv"
results_df.to_csv(per_case_path, index=False)
print("Saved:", per_case_path)

threshold_summary_path = OUTPUT_DIR / "gradcam_threshold_sensitivity.csv"
threshold_summary_df.to_csv(threshold_summary_path, index=False)
print("Saved:", threshold_summary_path)

summary_path = OUTPUT_DIR / "gradcam_summary_with_ci.csv"
gradcam_summary_df.to_csv(summary_path, index=False)
print("Saved:", summary_path)

# Save exemplar CAM volumes (best/median/worst), or all if SAVE_ALL_CAM_VOLUMES.
if saved_cam_volumes:
    cam_dir = OUTPUT_DIR / "cam_volumes"
    cam_dir.mkdir(exist_ok=True)
    for study_id, cam in saved_cam_volumes.items():
        np.save(cam_dir / f"{study_id}_cam.npy", cam.astype(np.float32))
    print(f"Saved {len(saved_cam_volumes)} CAM volume(s) to:", cam_dir)

print("\nAll Grad-CAM outputs saved to:", OUTPUT_DIR)
print(
    "\nRemember: download these from the Output tab before the session ends "
    "-- /kaggle/working/ is not guaranteed to persist across sessions."
)


## Conclusion

This notebook generated 3D Grad-CAM attention maps for the Run 2 classifier's PDAC-positive test predictions and quantitatively evaluated whether the model's attention localizes to the annotated tumor region.

**Results to carry into the thesis:**
- Mean IoU and Dice at the headline threshold (T=0.5), with 95% bootstrap CIs, plus the full threshold sensitivity curve (Cell 8) -- report both, not just the single headline number.
- Pointing Game accuracy, as a threshold-independent complement.
- The qualitative best/median/worst examples (Cell 10), which give a reader a concrete sense of what "IoU = X" or "Dice = Y" actually looks like on real cases -- important for an explainability chapter, since aggregate numbers alone don't communicate whether attention errors are minor (slightly off-center) or major (attending to unrelated anatomy).

**Limitations to state explicitly in the thesis:**
- IoU and Dice depend on the chosen binarization threshold; this notebook reports a sweep rather than a single value for exactly this reason, but the headline number chosen for prominent reporting (T=0.5) is still a specific, stated choice, not the only valid one.
- Grad-CAM's spatial resolution is limited by the target layer's downsampled feature map (upsampled via trilinear interpolation back to full resolution), which produces inherently coarse, blob-like localization rather than pixel-precise boundaries -- a well-known, general property of Grad-CAM, not specific to this implementation.
- Evaluation covers only PDAC-positive test cases with an intact tumor annotation; it says nothing about *why* the model correctly rejects non-PDAC cases, which is a different (and harder to visualize) question.
- As noted in the introduction, any test case affected by the Notebook 18 lateral-boundary tumor truncation has a ground-truth mask reflecting only the retained tumor extent -- check the per-case results (`gradcam_per_case_results.csv`) for these specific study IDs if reporting per-case detail.

This completes the two core components of the trustworthiness evaluation (calibration and explanation faithfulness) for the Run 2 baseline.
